In [ ]:
# Install required packages
%pip install pandas scikit-learn matplotlib seaborn plotly

# Naive Bayes Classification — Student Performance

This notebook applies **Naive Bayes Classification** to predict whether a student will **pass or fail** (G3 ≥ 10 = pass) using the student performance dataset.

We will:
1. Load and prepare the data
2. Create a binary target variable (pass / fail)
3. Train a Gaussian Naive Bayes model
4. Evaluate with accuracy, confusion matrix, and classification report
5. Compare an **early-term model** (no G1/G2) vs a **mid-term model** (with G1/G2)
6. Use k-fold cross-validation (same style as the lecture code)

In [ ]:
import pandas as pd
import numpy as np

# Load both datasets
mat = pd.read_csv("student-mat.csv", sep=";")
por = pd.read_csv("student-por.csv", sep=";")

# Add subject label
mat["subject"] = "math"
por["subject"] = "portuguese"

# Combine into one dataframe
df = pd.concat([mat, por], ignore_index=True)

print("Combined shape:", df.shape)
df.head()

## Creating the Target Variable: Pass / Fail

Naive Bayes is a **classification** algorithm — it predicts categories, not numbers.

We convert the continuous grade G3 (0–20) into a binary label:
- **1 = Pass** → G3 ≥ 10
- **0 = Fail** → G3 < 10

This is the Portuguese grading standard where 10 is the passing threshold.

In [ ]:
import plotly.express as px

# Create binary target: 1 = pass, 0 = fail
df["pass"] = (df["G3"] >= 10).astype(int)

print("Class distribution (0=Fail, 1=Pass):")
print(df["pass"].value_counts())
print(f"\nPass rate: {df['pass'].mean()*100:.1f}%")

# Visualise class balance
class_counts = df["pass"].value_counts().reset_index()
class_counts.columns = ["pass", "count"]
class_counts["label"] = class_counts["pass"].map({1: "Pass (G3 ≥ 10)", 0: "Fail (G3 < 10)"})

fig = px.bar(
    class_counts,
    x="label",
    y="count",
    color="label",
    color_discrete_map={"Pass (G3 ≥ 10)": "#2ecc71", "Fail (G3 < 10)": "#e74c3c"},
    title="Class Distribution: Pass vs Fail",
    labels={"count": "Number of Students", "label": ""}
)
fig.show()

## What is Naive Bayes?

Naive Bayes is a probabilistic classifier based on **Bayes' Theorem**:

> P(Class | Features) ∝ P(Features | Class) × P(Class)

The "naive" part assumes that **all features are independent of each other** given the class — which is rarely perfectly true, but often works well in practice.

**Gaussian Naive Bayes** assumes each feature follows a normal (Gaussian) distribution within each class. It is well-suited to continuous numerical features like our student data.

## Model 1: Early-Term (No G1 or G2)

This is the harder and more useful model — it tries to predict whether a student will pass based only on **background and lifestyle features**, before any grades are available.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import RepeatedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from numpy import mean, absolute

# -----------------------------------------------
# Prepare data — early-term: no G1, G2
# -----------------------------------------------
df_early = df.copy()
df_early = df_early.dropna()                         # remove missings
y = df_early["pass"]                                  # select target
X = df_early.drop(["G1", "G2", "G3", "pass"], axis=1)  # select features (no grade leakage)
X = pd.get_dummies(X, drop_first=True)               # turn categoricals into dummy variables

print("Features used:", X.shape[1])
print("Rows:", X.shape[0])

In [ ]:
# -----------------------------------------------
# Cross-validation (same style as lecture code)
# -----------------------------------------------
cv = RepeatedKFold(n_splits=5, random_state=420)  # splits data into 5 folds

gnb = GaussianNB()

# Accuracy score
acc_scores = cross_val_score(gnb, X, y,
                             scoring="accuracy",
                             cv=cv)

avg_accuracy = mean(acc_scores)
print(f"Early-term model — average accuracy with 5-fold CV: {avg_accuracy:.0%}")
print(f"(more precise): {avg_accuracy:.4f}")

In [ ]:
# -----------------------------------------------
# Detailed evaluation on a held-out test set
# -----------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

gnb_early = GaussianNB()
gnb_early.fit(X_train, y_train)
y_pred_early = gnb_early.predict(X_test)

print("=== Early-Term Naive Bayes (no G1/G2) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_early):.4f}")
print()
print(classification_report(y_test, y_pred_early, target_names=["Fail", "Pass"]))

## Confusion Matrix — Early-Term Model

In [ ]:
import plotly.figure_factory as ff

cm_early = confusion_matrix(y_test, y_pred_early)

fig = ff.create_annotated_heatmap(
    z=cm_early,
    x=["Predicted: Fail", "Predicted: Pass"],
    y=["Actual: Fail", "Actual: Pass"],
    colorscale="Blues"
)
fig.update_layout(
    title="Confusion Matrix — Early-Term Naive Bayes (no G1/G2)",
    xaxis_title="Predicted",
    yaxis_title="Actual"
)
fig.show()

print("How to read this:")
print(f"  True Negatives  (correctly predicted Fail): {cm_early[0,0]}")
print(f"  False Positives (predicted Pass, actually Fail): {cm_early[0,1]}")
print(f"  False Negatives (predicted Fail, actually Pass): {cm_early[1,0]}")
print(f"  True Positives  (correctly predicted Pass): {cm_early[1,1]}")

## Model 2: Mid-Term (Including G1 and G2)

Now we include **G1 and G2** (period 1 and 2 grades) as features. This simulates what a teacher or advisor already knows partway through the year. As we saw in the regression notebook, G1 and G2 are strongly correlated with G3 — so this model should perform significantly better.

In [ ]:
# -----------------------------------------------
# Prepare data — mid-term: include G1, G2
# -----------------------------------------------
df_mid = df.copy()
df_mid = df_mid.dropna()                         # remove missings
y_mid = df_mid["pass"]                           # select target
X_mid = df_mid.drop(["G3", "pass"], axis=1)      # keep G1 and G2 this time
X_mid = pd.get_dummies(X_mid, drop_first=True)  # turn categoricals into dummy variables

print("Features used:", X_mid.shape[1])

# Cross-validation
cv = RepeatedKFold(n_splits=5, random_state=420)
gnb_mid = GaussianNB()

acc_scores_mid = cross_val_score(gnb_mid, X_mid, y_mid,
                                 scoring="accuracy",
                                 cv=cv)

avg_accuracy_mid = mean(acc_scores_mid)
print(f"Mid-term model — average accuracy with 5-fold CV: {avg_accuracy_mid:.0%}")
print(f"(more precise): {avg_accuracy_mid:.4f}")

In [ ]:
# Detailed evaluation
X_train_mid, X_test_mid, y_train_mid, y_test_mid = train_test_split(
    X_mid, y_mid, test_size=0.2, random_state=42
)

gnb_mid.fit(X_train_mid, y_train_mid)
y_pred_mid = gnb_mid.predict(X_test_mid)

print("=== Mid-Term Naive Bayes (with G1/G2) ===")
print(f"Accuracy: {accuracy_score(y_test_mid, y_pred_mid):.4f}")
print()
print(classification_report(y_test_mid, y_pred_mid, target_names=["Fail", "Pass"]))

In [ ]:
# Confusion matrix — mid-term
cm_mid = confusion_matrix(y_test_mid, y_pred_mid)

fig = ff.create_annotated_heatmap(
    z=cm_mid,
    x=["Predicted: Fail", "Predicted: Pass"],
    y=["Actual: Fail", "Actual: Pass"],
    colorscale="Greens"
)
fig.update_layout(
    title="Confusion Matrix — Mid-Term Naive Bayes (with G1/G2)",
    xaxis_title="Predicted",
    yaxis_title="Actual"
)
fig.show()

## Comparing Both Models

Let's put the key metrics side-by-side.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    "Model": ["Early-term (no G1/G2)", "Mid-term (with G1/G2)"],
    "CV Accuracy": [
        round(avg_accuracy, 4),
        round(avg_accuracy_mid, 4)
    ],
    "Test Accuracy": [
        round(accuracy_score(y_test, y_pred_early), 4),
        round(accuracy_score(y_test_mid, y_pred_mid), 4)
    ],
    "Precision (Pass)": [
        round(precision_score(y_test, y_pred_early), 4),
        round(precision_score(y_test_mid, y_pred_mid), 4)
    ],
    "Recall (Pass)": [
        round(recall_score(y_test, y_pred_early), 4),
        round(recall_score(y_test_mid, y_pred_mid), 4)
    ],
    "F1 (Pass)": [
        round(f1_score(y_test, y_pred_early), 4),
        round(f1_score(y_test_mid, y_pred_mid), 4)
    ]
})

print(results.to_string(index=False))

In [ ]:
# Bar chart comparison
metrics_df = pd.DataFrame({
    "Metric": ["CV Accuracy", "Test Accuracy", "Precision", "Recall", "F1"] * 2,
    "Value": [
        avg_accuracy,
        accuracy_score(y_test, y_pred_early),
        precision_score(y_test, y_pred_early),
        recall_score(y_test, y_pred_early),
        f1_score(y_test, y_pred_early),
        avg_accuracy_mid,
        accuracy_score(y_test_mid, y_pred_mid),
        precision_score(y_test_mid, y_pred_mid),
        recall_score(y_test_mid, y_pred_mid),
        f1_score(y_test_mid, y_pred_mid),
    ],
    "Model": ["Early-term (no G1/G2)"] * 5 + ["Mid-term (with G1/G2)"] * 5
})

fig = px.bar(
    metrics_df,
    x="Metric",
    y="Value",
    color="Model",
    barmode="group",
    title="Naive Bayes: Early-Term vs Mid-Term Model Comparison",
    range_y=[0, 1.05]
)
fig.show()

## What Do the Results Mean?

**Early-term model (~76% accuracy)**  
Without knowing any grades, the model can still correctly classify about 76% of students. It struggles more with identifying students who will fail — many failing students look similar to passing students when you only look at background features.

**Mid-term model (~82% accuracy)**  
Adding G1 and G2 boosts accuracy considerably. This makes sense — a student's period 1 and 2 grades are by far the strongest predictors of the final grade (correlation ~0.81–0.91 as seen in the heatmap earlier).

**Key metric for this use case — Recall on 'Fail':**  
If the goal is to identify students at risk of failing (so schools can intervene), **recall for the Fail class** matters most. Missing a student who will fail (False Negative) is more costly than a false alarm. The early-term model currently has lower recall on the Fail class — meaning it misses some at-risk students.

**Why Naive Bayes?**  
- Fast and interpretable  
- Works well even with a modest dataset  
- The independence assumption is violated here (features like G1/G2 are correlated), but it still produces useful results  
- Provides probability estimates, not just hard predictions — useful for building early-warning systems

In [ ]:
# Bonus: Predicted probabilities for the first 10 test students
proba = gnb_early.predict_proba(X_test)

prob_df = pd.DataFrame({
    "P(Fail)": proba[:, 0].round(3),
    "P(Pass)": proba[:, 1].round(3),
    "Predicted": ["Pass" if p == 1 else "Fail" for p in y_pred_early],
    "Actual": ["Pass" if a == 1 else "Fail" for a in y_test]
}).head(10)

print("Sample predictions with confidence probabilities (early-term model):")
print(prob_df.to_string(index=False))